In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import pickle

print("Libraries imported!")

Libraries imported!


In [2]:
df = pd.read_csv('../data/processed_matches.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSample:")
df.head()

Shape: (1169, 9)

Columns: ['team1', 'team2', 'toss_winner', 'toss_decision', 'venue', 'venue_win_rate', 'team1_recent_form', 'team2_recent_form', 'team1_win']

Sample:


,team1,team2,toss_winner,toss_decision,venue,venue_win_rate,team1_recent_form,team2_recent_form,team1_win
0,6,13,13,1,23,0.285714,0.5,0.5,1
1,0,10,0,0,41,0.500000,0.5,0.5,1
2,11,2,11,0,16,0.500000,0.5,0.5,0
3,8,13,8,0,56,0.611111,0.5,0.0,0
4,1,6,1,0,14,0.000000,0.5,0.5,0


In [3]:
X = df.drop('team1_win', axis=1)
y = df['team1_win']

print("Features (X):", X.shape)
print("Target (y):", y.shape)
print("\nFeature columns:", X.columns.tolist())

Features (X): (1169, 8)
Target (y): (1169,)

Feature columns: ['team1', 'team2', 'toss_winner', 'toss_decision', 'venue', 'venue_win_rate', 'team1_recent_form', 'team2_recent_form']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

Training data: (935, 8)
Testing data : (234, 8)


In [5]:
# Saare models define karo
models = {
    'Logistic Regression'  : LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest'        : RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost'              : XGBClassifier(random_state=42, eval_metric='logloss'),
    'Gradient Boosting'    : GradientBoostingClassifier(random_state=42),
}

# Saare models train karo aur accuracy store karo
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred     = model.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    results[name] = accuracy
    print(f"{name:25} : {round(accuracy*100, 2)}%")

Logistic Regression       : 74.79%
Random Forest             : 68.8%
XGBoost                   : 65.81%
Gradient Boosting         : 70.09%


In [6]:
# Voting Classifier — top 3 models ka combination
voting = VotingClassifier(
    estimators=[
        ('lr', models['Logistic Regression']),
        ('gb', models['Gradient Boosting']),
        ('rf', models['Random Forest']),
    ],
    voting='soft'
)
voting.fit(X_train, y_train)
voting_pred     = voting.predict(X_test)
voting_accuracy = accuracy_score(y_test, voting_pred)
print(f"Voting Classifier         : {round(voting_accuracy*100, 2)}%")

# Results update karo
results['Voting Classifier'] = voting_accuracy

# Final comparison table
results_df = pd.DataFrame({
    'Model'   : list(results.keys()),
    'Accuracy': [round(v*100, 2) for v in results.values()]
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("\nFINAL COMPARISON:")
print(results_df)

Voting Classifier         : 71.37%

FINAL COMPARISON:
                 Model  Accuracy
0  Logistic Regression     74.79
1    Voting Classifier     71.37
2    Gradient Boosting     70.09
3        Random Forest     68.80
4              XGBoost     65.81


In [7]:
# Best model find karo automatically
best_model_name = results_df.iloc[0]['Model']
best_accuracy   = results_df.iloc[0]['Accuracy']

print(f"Best Model : {best_model_name}")
print(f"Accuracy   : {best_accuracy}%")

# Sahi model object lo
all_models = {
    'Logistic Regression' : models['Logistic Regression'],
    'Random Forest'       : models['Random Forest'],
    'XGBoost'             : models['XGBoost'],
    'Gradient Boosting'   : models['Gradient Boosting'],
    'Voting Classifier'   : voting
}

best_model = all_models[best_model_name]

# Save karo
with open('../model/ipl_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print(f"\nModel saved: ipl_model.pkl")

Best Model : Logistic Regression
Accuracy   : 74.79%

Model saved: ipl_model.pkl


In [8]:
print("=" * 50)
print("MODULE 3 SUMMARY — Model Training")
print("=" * 50)
print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")
print(f"Total features   : {X_train.shape[1]}")
print()
print("FEATURES USED:")
for col in X.columns:
    print(f"  - {col}")
print()
print("MODEL COMPARISON:")
for _, row in results_df.iterrows():
    marker = " ← Best" if row['Model'] == best_model_name else ""
    print(f"  {row['Model']:25} : {row['Accuracy']}%{marker}")
print()
print(f"Best Model : {best_model_name}")
print(f"Accuracy   : {best_accuracy}%")
print()
print("KEY INSIGHT:")
print("Feature engineering se accuracy")
print("54% se 74% ho gayi — 20% jump!")
print("Better data > Better model")
print()
print("Model saved: ipl_model.pkl")
print("=" * 50)
print("NEXT: app.py — Streamlit Deployment")

MODULE 3 SUMMARY — Model Training
Training samples : 935
Testing samples  : 234
Total features   : 8

FEATURES USED:
  - team1
  - team2
  - toss_winner
  - toss_decision
  - venue
  - venue_win_rate
  - team1_recent_form
  - team2_recent_form

MODEL COMPARISON:
  Logistic Regression       : 74.79% ← Best
  Voting Classifier         : 71.37%
  Gradient Boosting         : 70.09%
  Random Forest             : 68.8%
  XGBoost                   : 65.81%

Best Model : Logistic Regression
Accuracy   : 74.79%

KEY INSIGHT:
Feature engineering se accuracy
54% se 74% ho gayi — 20% jump!
Better data > Better model

Model saved: ipl_model.pkl
NEXT: app.py — Streamlit Deployment
